# Day 09 - Talking to APIs over HTTP
Vidya Sankalp | Applied GenAI Engineering

> **The big idea:** most useful data does not live inside your program - it lives on another computer somewhere on the internet (a *server*). To get it, your program sends a **request**, the request travels across the internet, and the server sends back a **response**. That conversation is called **HTTP**.

> **Today you call a REAL server.** We use **DummyJSON** (`https://dummyjson.com`) - a free, real API running on the internet that serves shopping data: products, categories, reviews, carts. You send real requests; a real machine replies. You learn GET, the address (URL), status codes, JSON, query filters, POST, headers, timeouts, and handling errors.

> **Why this comes before async:** once you have made real calls that take real time over the internet, the next day (async) has a genuine problem to make faster. No pretend `sleep()`.

> **Library:** `httpx` - a friendly modern way to make HTTP calls in Python, and the exact library your real project uses.

> **You need internet for this notebook** (the calls go to a real server). If a cell ever fails with a connection error, just check your wifi and run it again.


---


## 0 - One-time setup

We only need one library. Run this once.

**The shape of every HTTP call you'll write today:**
```
httpx.get(ADDRESS, params={...}, timeout=10.0)
        |          |              |
        |          |              +-- how many SECONDS to wait before giving up
        |          +-- optional extra options added after a '?' in the address
        +-- the web address (URL) of what you want
```
Every call returns a **response** object. From it you read `.status_code` (a number: did it work?) and `.json()` (the actual data, as a Python dict). Those three - the call, the status, the data - are the whole skill.


In [ ]:
# We only need the 'httpx' library to make HTTP calls, and 'time' to measure how long calls take.
import httpx      # httpx = the tool that sends requests over the internet and gives us the reply
import time       # time = used later to measure how many seconds a call takes

# BASE is the 'home address' of the server we will talk to. We keep it in a variable so we don't
# have to retype 'https://dummyjson.com' on every single call - we just write f'{BASE}/products'.
BASE = 'https://dummyjson.com'

# Let's check we can actually reach the server before we start.
# httpx.get(URL) sends a GET request (a 'please give me data' request) to that web address.
#   timeout=10.0  ->  wait at most 10 SECONDS for a reply. If the server hasn't answered by then,
#                     httpx stops waiting and raises an error instead of freezing forever.
#                     (Without a timeout, a stuck server could freeze your program indefinitely.)
r = httpx.get(f'{BASE}/test', timeout=10.0)

print('Reached the server. Status:', r.status_code)   # .status_code is a number telling us how it went (200 = OK)
print('Server says:', r.json())                       # .json() turns the reply text into a Python dictionary


---


---


## The Four Things You Can Ask a Server to Do

Almost every API call is one of four **verbs** - the *action* you want. You have met GET and POST; now we complete the set.

| Verb | Plain meaning | Everyday picture | Example |
|------|---------------|------------------|---------|
| **GET** | read / fetch something | looking something up | `GET /products/1` |
| **POST** | create something new | filling in a brand-new form | `POST /products/add` |
| **PUT** | update something that exists | editing a form you already submitted | `PUT /products/1` |
| **DELETE** | remove something | throwing something away | `DELETE /products/1` |

In `httpx` each one is just a method with the same shape you already know: `httpx.get(...)`, `httpx.post(...)`, `httpx.put(...)`, `httpx.delete(...)`.


## 1 - Your First Request: GET

Asking a server for data is a **GET** request - you are *getting* something. You send the **address** of what you want, and the server sends back the data. Let's ask for a list of products.


In [ ]:
# Send a GET request asking for the list of products.
#   f'{BASE}/products'  ->  the full address: https://dummyjson.com/products
#   timeout=10.0        ->  wait at most 10 seconds for the reply, then give up (see setup cell)
response = httpx.get(f'{BASE}/products', timeout=10.0)

# .status_code is a 3-digit number the server sends back to say how the request went.
#   200 means 'OK - here is your data'. (We cover the other numbers in the next section.)
print('Status code:', response.status_code)

# The reply arrives as plain TEXT. .json() converts that text into a real Python dictionary
# so we can pull values out of it with square brackets, just like any dict.
data = response.json()

print('Total products on the server:', data['total'])    # 'total' = how many products exist in all
print('How many we got back:', len(data['products']))    # the server sends them in pages (30 at a time by default)
print('First product:', data['products'][0]['title'])    # data['products'] is a list; [0] = the first one


Three things just happened - the whole game:
1. **`httpx.get(...)`** - we asked the real server for products.
2. **`.status_code`** - the server told us how it went. `200` = success.
3. **`.json()`** - the reply arrived as text; `.json()` turns it into a normal Python dict.

> The server returned `total`, `skip`, and `limit` too. By default it sends 30 products at a time (this is called *pagination* - sending big lists in pages).


In [ ]:
# Ask again, but this time use a query option to get only 5 products (explained in section 3).
#   params={'limit': 5}  ->  httpx turns this into  ?limit=5  on the end of the address,
#                            which tells the server 'only send 5 products please'.
data = httpx.get(f'{BASE}/products', params={'limit': 5}, timeout=10.0).json()

# Loop over the list of products and print a few fields from each one.
# Each 'p' is one product dictionary with keys like 'id', 'title', 'category', 'price'.
for p in data['products']:
    print(f"  #{p['id']}  {p['title']:<32} {p['category']:<18} ${p['price']}")


---


## 2 - Status Codes: Did It Work?

Every response carries a **status code** - a 3-digit number saying how it went. You only need a handful:

| Code | Meaning | Plain words |
|------|---------|-------------|
| **200** | OK | 'Here's your data.' |
| **404** | Not Found | 'That thing does not exist.' |
| **400 / 422** | Bad Request | 'You asked for something invalid.' |
| **500** | Server Error | 'Something broke on my end.' |

Memory aid: **2xx = all good**, **4xx = your fault** (asked wrong), **5xx = their fault** (server broke).


In [ ]:
# Ask for a product that EXISTS (id 1). The server finds it and replies with status 200 (OK).
r1 = httpx.get(f'{BASE}/products/1', timeout=10.0)   # /products/1 = 'the product whose id is 1'
print('Product 1     :', r1.status_code, '->', r1.json()['title'])

# Ask for a product that does NOT exist (id 99999). The server can't find it,
# so it replies with status 404 (Not Found) instead of 200.
r2 = httpx.get(f'{BASE}/products/99999', timeout=10.0)
print('Product 99999 :', r2.status_code, '->', r2.json())


In [ ]:
# It is good habit to CHECK the status code before trusting the data.
r = httpx.get(f'{BASE}/products/99999', timeout=10.0)

if r.status_code == 200:                 # 200 = success, the data is there
    print('Got it:', r.json()['title'])
elif r.status_code == 404:               # 404 = the thing we asked for does not exist
    print('That product does not exist - show a friendly message instead.')
else:                                    # anything else = some other problem
    print('Something unexpected:', r.status_code)

# Shortcut: instead of checking by hand, .raise_for_status() will AUTOMATICALLY raise an
# error if the status is a failure code (4xx or 5xx). You can then catch it with try/except.
try:
    httpx.get(f'{BASE}/products/99999', timeout=10.0).raise_for_status()
except httpx.HTTPStatusError as e:       # this 'except' runs only if the status was a failure
    print('raise_for_status caught:', e.response.status_code)


---


## 3 - Two Ways to Ask for Specific Data

### (a) Part of the address - the **path**
To get *one specific* product, put its id in the address: `/products/1`. The `1` is part of the path.

### (b) After a `?` - **query options**
To change *how* the list comes back, add options after a `?`. For example `/products?limit=5` means 'just 5 please'. `httpx` builds this for you with `params={...}`.


In [ ]:
# (a) PATH parameter - the id is part of the address itself.
#     /products/2  means 'the one product whose id is 2'.
p = httpx.get(f'{BASE}/products/2', timeout=10.0).json()
print('One product:', p['title'], '-', p['category'], '- $' + str(p['price']))

print()
# (b) QUERY parameters - extra options after a '?' that change HOW the list comes back.
#     params={...}  ->  httpx builds the '?...' part of the address for us.
#       'limit': 3            ->  send only 3 products
#       'select': 'title,price' ->  for each product, include ONLY the title and price fields
#     So the real address becomes:  /products?limit=3&select=title,price
data = httpx.get(f'{BASE}/products',
                 params={'limit': 3, 'select': 'title,price'},
                 timeout=10.0).json()
for item in data['products']:
    print('  ', item)


> `params={'limit': 3, 'select': 'title,price'}` becomes `?limit=3&select=title,price` in the address. Letting `httpx` build it keeps things clean and handles special characters safely.


In [ ]:
# Searching is just another query option. The '/products/search' endpoint looks for matching text.
#   params={'q': 'phone'}  ->  becomes  ?q=phone  ('q' is short for 'query', the thing to search for)
found = httpx.get(f'{BASE}/products/search', params={'q': 'phone'}, timeout=10.0).json()

print(f"Found {found['total']} products matching 'phone'. First few:")
for p in found['products'][:5]:          # [:5] = just the first 5 from the list
    print('  ', p['title'])


---


## 4 - Explore the Whole Store

Now that you can call endpoints, look at the other kinds of data this real server offers.


In [ ]:
# Ask the server for the list of all category NAMES.
# This endpoint returns a plain list of strings (not a dict with 'products' inside).
cats = httpx.get(f'{BASE}/products/category-list', timeout=10.0).json()

print('There are', len(cats), 'categories. Some of them:')
print('  ', cats[:10])                   # [:10] = first 10 category names


In [ ]:
# Get every product in ONE category. Here the category name is part of the PATH:
#   /products/category/smartphones  ->  'all products in the smartphones category'
smartphones = httpx.get(f'{BASE}/products/category/smartphones', timeout=10.0).json()

print(f"smartphones category has {smartphones['total']} products:")
for p in smartphones['products'][:5]:    # show just the first 5
    print(f"  {p['title']:<28} ${p['price']}")


In [ ]:
# On this server, reviews are not a separate endpoint - they come INSIDE each product,
# under the 'reviews' key. So we fetch the product, then read its 'reviews' list.
product = httpx.get(f'{BASE}/products/1', timeout=10.0).json()

print('Reviews for:', product['title'])
for rev in product['reviews']:           # each 'rev' is one review dict
    # build a star string: filled stars for the rating, empty stars for the rest (out of 5)
    stars = '★' * rev['rating'] + '☆' * (5 - rev['rating'])
    print(f"  {stars}  {rev['reviewerName']}: {rev['comment']}")


In [ ]:
# A 'cart' here works like an order: it has a list of products, each with a quantity and a total.
cart = httpx.get(f'{BASE}/carts/1', timeout=10.0).json()   # /carts/1 = the cart whose id is 1

print('Cart', cart['id'], '- total $' + str(cart['total']), 'for', cart['totalProducts'], 'products')
for item in cart['products'][:4]:        # show the first 4 items in the cart
    print(f"  {item['title']:<28} x{item['quantity']}  = ${item['total']}")


---


## 5 - Sending Data: POST

GET *gets* data. **POST** *sends* data to the server - like submitting a form. To add a product, we POST its details in the request **body** (a little JSON package), using `json=...`.

> DummyJSON *simulates* the add - it replies as if it saved, and gives the new item an id, but it doesn't really change the server. That's perfect for practice.


In [ ]:
# To CREATE something we use POST and send a 'body' - a little package of data we want to add.
# We put that data in a normal Python dictionary first.
new_product = {'title': 'ShopSmart Travel Mug', 'price': 14.99, 'category': 'kitchen-accessories'}

# httpx.post(URL, json=...) sends a POST request.
#   json=new_product  ->  httpx converts our dict into JSON text and puts it in the request body.
#                         (For GET we used params=; for sending data with POST we use json=.)
#   timeout=10.0      ->  wait at most 10 seconds (same as before)
r = httpx.post(f'{BASE}/products/add', json=new_product, timeout=10.0)

print('Status:', r.status_code)          # a 2xx number (200 or 201) means success
created = r.json()
print('Server created it with new id:', created['id'])   # the server assigns a brand-new id
print('Title it stored:', created['title'])


> A successful POST usually comes back as **200 (OK)** or **201 (Created)** - both are '2xx = good'. `201` specifically means 'I created the thing you sent.' The server also gave our new product an `id`.


---


---


## 6 - Updating and Removing: PUT and DELETE

### PUT - change something that already exists
**POST** makes a *new* thing. **PUT** *updates* a thing that is already there. You send the fields you want to change in the body, and the server sends back the updated item.

> **Safe to practise:** DummyJSON only *simulates* PUT - it replies as if it updated, but nothing on the real server actually changes. So you can run this as many times as you like with no worries.


In [ ]:
# PUT is for UPDATING something that already exists. We send only the field(s) we want to change.
#   f'{BASE}/products/1'   ->  update the product whose id is 1
#   json={'price': 9.99}   ->  the change we want: set its price to 9.99
#   timeout=10.0           ->  wait at most 10 seconds
r = httpx.put(f'{BASE}/products/1', json={'price': 9.99}, timeout=10.0)

print('Status:', r.status_code)          # 200 = the update was accepted
updated = r.json()
print('Product :', updated['title'])
print('New price:', updated['price'])     # the field we changed comes back updated
# (Reminder: DummyJSON only SIMULATES this - the real server data does not actually change.)


> You'll also see **PATCH**, which is very similar. The difference in plain words: **PUT** = 'here is the updated thing', **PATCH** = 'here is just the small part to change'. In everyday use they're often used the same way, and `httpx` has `httpx.patch(...)` too.


### DELETE - remove something
**DELETE** asks the server to remove a thing. DummyJSON replies with the item it 'deleted', plus two extra labels - `isDeleted` and `deletedOn` - so you can confirm it worked.

> Again, this is **simulated** - the product is not really removed from the server. Safe to run freely.


In [ ]:
# DELETE asks the server to remove something. We don't send a body - we just name what to delete.
#   f'{BASE}/products/1'  ->  delete the product whose id is 1
#   timeout=10.0          ->  wait at most 10 seconds
r = httpx.delete(f'{BASE}/products/1', timeout=10.0)

print('Status:', r.status_code)          # 200 = the delete request was handled
deleted = r.json()
print('Deleted product:', deleted['title'])
print('isDeleted flag :', deleted['isDeleted'])     # the server adds this flag to confirm: True
print('Deleted at     :', deleted['deletedOn'])     # ...and a timestamp of when it 'happened'
# (Reminder: simulated only - nothing is really removed from the server.)


Notice the shape never changed across all four verbs: you call a method (`get`/`post`/`put`/`delete`), check `.status_code`, and read `.json()`. Learn it once, use it everywhere.


## 7 - Two Practical Extras: Headers and Timeouts

### Headers - extra labels on a request/response
Every request/response carries **headers** - small labels with extra info, like what kind of content it is.

### Timeouts - don't wait forever
If the internet is slow or the server is stuck, you don't want your program frozen. A **timeout** says 'give up after N seconds'. We've been passing `timeout=10.0` on every call - that's it in action.


In [ ]:
# Every response also carries HEADERS - small labels of extra info about the reply.
r = httpx.get(f'{BASE}/products/1', timeout=10.0)

# .headers is a dictionary-like object. 'content-type' tells us what KIND of data the body is.
print('Content-Type header:', r.headers.get('content-type'))
# 'application/json' means the body is JSON text - which is exactly why .json() works on it.


In [ ]:
# Let's SEE a timeout actually fire. We give an impossibly short limit of 0.001 seconds.
#   timeout=0.001  ->  'give up if there's no reply within one-thousandth of a second'.
#                      No internet call is that fast, so httpx will give up and raise an error.
try:
    httpx.get(f'{BASE}/products', timeout=0.001)
    print('Got a response (network was extremely fast)')
except httpx.TimeoutException:           # this runs when the call takes longer than the timeout
    print('Timed out - the call took longer than our tiny limit, so httpx gave up.')


---


## 8 - A Slow Call (remember this for tomorrow)

DummyJSON lets us add a `delay` to any call (in milliseconds) - handy for practising with slow responses. Let's make a call that takes about **half a second**.

One slow call is fine. But what if you need *many*? Watch several **one after another** - this is exactly the problem we fix on the async day.


In [ ]:
# DummyJSON lets us add an artificial delay so we can practise with slow responses.
#   params={'delay': 500}  ->  becomes ?delay=500, telling the server 'wait 500 milliseconds
#                              (half a second) before replying'. Great for feeling real latency.
#   timeout=15.0           ->  we raise the timeout to 15s so our own limit doesn't fire first.
start = time.perf_counter()              # perf_counter() = a precise stopwatch (in seconds)
r = httpx.get(f'{BASE}/products/1', params={'delay': 500}, timeout=15.0)
print('Got:', r.json()['title'])
print(f'One slow call took {time.perf_counter()-start:.1f}s')   # subtract start = elapsed seconds


In [ ]:
# Now make FIVE slow calls one after another. Each waits ~0.5s, and because we do them
# one-by-one (each finishes before the next starts), the times ADD UP to about 2.5 seconds.
start = time.perf_counter()
for i in range(5):                       # i goes 0,1,2,3,4
    httpx.get(f'{BASE}/products/{i+1}', params={'delay': 500}, timeout=15.0)
print(f'Five slow calls, one after another, took {time.perf_counter()-start:.1f}s')
print('That is ~5 x 0.5s. Tomorrow, async runs them together in ~0.5s total!')


---


---


## 9 - Bonus: Talking to the OpenAI API (it's just HTTP!)

Here's the big reveal for everyone building AI apps: **calling an AI model like ChatGPT is just another HTTP request.** Same `httpx`, same `.status_code`, same `.json()` you have used all day. There is no magic - you POST a question, the server sends back an answer.

There is exactly **one new idea**: this server needs to know *who you are* and that you're allowed to use it. You prove that with an **API key**, sent in a special header.


### The new concept: authentication with a Bearer token

DummyJSON let anyone call it. Most real APIs do not - they need a **key** (a secret password) so they know who is calling and can bill them. You send the key in the **`Authorization` header** like this:

```
Authorization: Bearer YOUR_API_KEY_HERE
```

The word **`Bearer`** just means 'the bearer (holder) of this key is allowed in'. This is the single most common way APIs handle keys, so once you know it, you can call thousands of different APIs.

> **Keep your key secret.** Never paste it into code you share, never commit it to GitHub. A leaked key can run up real charges. Below we read it from an environment variable instead of typing it in the notebook.


### Step 1 - Set up your key (each student uses their own)

Get a key from <https://platform.openai.com/api-keys>. Then set it as an environment variable **before** starting Jupyter:

- Mac/Linux: `export OPENAI_API_KEY="sk-..."`
- Windows (PowerShell): `setx OPENAI_API_KEY "sk-..."`

The cell below reads it safely. If it isn't set, the rest of this section simply skips - the rest of the notebook still works without a key.


In [ ]:
import os, httpx, json, time

# We read the secret API key from an ENVIRONMENT VARIABLE instead of typing it in the notebook.
#   os.environ.get('OPENAI_API_KEY')  ->  looks up a variable named OPENAI_API_KEY that you set
#                                         in your terminal before starting Jupyter. Returns the
#                                         key string if it exists, or None if you never set it.
# Why not just paste the key here? Because anyone who sees your notebook would see your secret
# key and could run up charges on your account. Keeping it in the environment keeps it private.
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

# This is the web address of OpenAI's chat endpoint - the 'door' we POST our question to.
OPENAI_URL = 'https://api.openai.com/v1/chat/completions'

# bool(...) turns the key into True (key exists) or False (key is None). We use this flag below
# so that every OpenAI cell can SKIP itself politely if you haven't set a key.
have_key = bool(OPENAI_API_KEY)
print('API key found - the OpenAI cells will run.' if have_key
      else 'No API key set - OpenAI cells will be skipped (everything else still works).')


### Step 2 - Your first AI call (a chat completion)

You send a list of **messages** (the conversation) and the model replies. The request body has two key parts:
- `model` - which AI to use (e.g. `gpt-4o-mini`, a cheap fast one)
- `messages` - the conversation, each with a `role` (`system`, `user`, or `assistant`) and `content`

The reply lives at `response.json()['choices'][0]['message']['content']`.


In [ ]:
if have_key:                              # only run if a key was found (see previous cell)
    response = httpx.post(                # POST because we are SENDING a question to the AI
        OPENAI_URL,                       # the chat endpoint address
        headers={                         # headers = extra labels attached to the request
            'Authorization': f'Bearer {OPENAI_API_KEY}'
            # ^ THE NEW IDEA: this header proves who you are. The format is the word 'Bearer',
            #   a space, then your key. 'Bearer' just means 'the holder of this key is allowed in'.
            #   Almost every paid API uses this exact header to check you're allowed.
        },
        json={                            # json= the body: the data we send, as a dict
            'model': 'gpt-4o-mini',       # which AI model to use (this one is small, fast, cheap)
            'messages': [                 # the conversation, as a list of messages
                # each message has a 'role' and the text 'content':
                {'role': 'system', 'content': 'You are a helpful ShopSmart assistant.'},  # sets behaviour
                {'role': 'user',   'content': 'Suggest a one-line slogan for a yoga mat.'}, # your question
            ],
        },
        timeout=30.0,    # wait up to 30 SECONDS - AI replies are slower than normal API calls,
                         # so we allow more time than the 10s we used for DummyJSON.
    )
    print('Status:', response.status_code)            # 200 = the AI answered successfully
    data = response.json()                            # turn the reply into a Python dict
    # The actual answer is buried inside the reply. We dig down to it:
    #   data['choices']         -> a list of possible answers (usually just one)
    #   [0]                     -> the first answer
    #   ['message']['content']  -> the actual text the AI wrote
    reply = data['choices'][0]['message']['content']
    print('AI reply:', reply)
else:
    print('(skipped - no API key)')


Look closely: that is the **exact same `httpx.post(...)` shape** as adding a review to DummyJSON. A URL, a headers dict, a `json=` body, then read `.json()`. The only additions are the `Authorization` header and OpenAI's particular body fields.


### Step 3 - How much did it cost? Read the token usage

AI APIs charge by **tokens** (roughly, pieces of words). Every reply tells you how many it used, in a `usage` block. You read it the same way you read any other field in the JSON.


In [ ]:
if have_key:
    # Every reply includes a 'usage' section telling you how many TOKENS were used.
    # A token is roughly a piece of a word. You are billed per token, so this is your 'meter'.
    usage = data['usage']                                   # 'data' came from the previous cell
    print('Tokens you sent (prompt)    :', usage['prompt_tokens'])      # tokens in YOUR question
    print('Tokens the AI sent back     :', usage['completion_tokens'])  # tokens in the ANSWER
    print('Total tokens (what you pay) :', usage['total_tokens'])       # the two added together

    # Rough cost = (input tokens x input price) + (output tokens x output price).
    # These per-token prices are just examples - always check OpenAI's current pricing page.
    #   0.15/1_000_000  means '$0.15 per MILLION input tokens', written as price-per-single-token.
    in_rate  = 0.15 / 1_000_000     # dollars per input token  (illustrative)
    out_rate = 0.60 / 1_000_000     # dollars per output token (illustrative)
    cost = usage['prompt_tokens'] * in_rate + usage['completion_tokens'] * out_rate
    print(f'Rough cost of this one call : ${cost:.6f}')     # :.6f = show 6 decimal places
else:
    print('(skipped - no API key)')


### Step 4 - Streaming: get the answer word by word

By default you wait for the *whole* answer, then see it. But ChatGPT-style apps show the reply appearing **as it's written**. You ask for that by adding `'stream': True` to the body. The server then sends the answer in many small pieces using **Server-Sent Events** - each line starts with `data: `.

> **This is your bridge to the next two days.** Streaming is exactly the `yield` / `async for` idea from the async day, and the `data: ...` format is what your project's chat endpoint produces. Here you *receive* it; later you *produce* it.


In [ ]:
if have_key:
    # Normally we wait for the WHOLE answer. 'streaming' means we get it in small pieces as the
    # AI writes it, so words appear live. httpx.stream(...) keeps the connection open for that.
    #   'POST'        ->  the request method (sending data, same as httpx.post)
    #   stream=True   ->  inside the body, this asks OpenAI to send the answer piece by piece
    with httpx.stream(
        'POST', OPENAI_URL,
        headers={'Authorization': f'Bearer {OPENAI_API_KEY}'},   # same auth header as before
        json={
            'model': 'gpt-4o-mini',
            'messages': [{'role': 'user', 'content': 'List 3 yoga mat features, very briefly.'}],
            'stream': True,                 # <- ask for the answer in pieces, not all at once
        },
        timeout=30.0,                       # up to 30s, like the other AI call
    ) as response:
        # The server sends many lines. Each useful line looks like:  data: {...some JSON...}
        for line in response.iter_lines():           # read the reply one line at a time
            if not line or not line.startswith('data: '):
                continue                              # skip blank lines and non-data lines
            chunk = line[len('data: '):]              # remove the 'data: ' prefix, keep the JSON part
            if chunk == '[DONE]':                     # OpenAI sends this exact text to mean 'finished'
                break
            piece = json.loads(chunk)                 # turn this piece of JSON text into a dict
            # each piece carries a tiny bit of the answer inside ['choices'][0]['delta']
            delta = piece['choices'][0]['delta']
            if 'content' in delta:                    # some pieces have no text - skip those
                print(delta['content'], end='', flush=True)  # end='' = no newline, so words join up
    print('\n(done streaming)')
else:
    print('(skipped - no API key)')


Notice the pattern: the server `yield`s pieces, each one a `data: {...}` line, ending with `data: [DONE]`. That is **exactly** the streaming format your project's chat uses - you have now seen both ends of it.


### Why this matters for your course

Every 'AI feature' you will build is, underneath, an HTTP call like the ones above:
- a POST with an `Authorization: Bearer` header,
- a `messages` list as the body,
- a JSON reply you read fields out of,
- optionally streamed piece by piece.

The fancy AI libraries (like the `openai` Python package) are just friendly wrappers around exactly this. Now that you've seen the raw HTTP, those libraries will never feel like magic.


## Quick Recap - Day 09

| You learned | How |
|-------------|-----|
| Ask a real server for data | `httpx.get('https://dummyjson.com/products')` |
| Check it worked | `.status_code` (200 ok, 201 created, 404 missing) |
| Read the reply | `.json()` turns it into a Python dict |
| Get one specific item | id in the path: `/products/1` |
| Control / filter a list | `params={'limit': 5, 'select': 'title,price'}` |
| Search | `/products/search` with `params={'q': 'phone'}` |
| Send data to a server | `httpx.post(url, json={...})` |
| Update something | `httpx.put(url, json={...})` |
| Remove something | `httpx.delete(url)` |
| See extra info | `.headers` |
| Don't wait forever | `timeout=10.0` |
| Make a call slow on purpose | `params={'delay': 500}` |

**Three things to remember:**
1. Every HTTP call is the same shape: send a request over the internet -> get a response (status code + body).
2. `2xx` = good, `4xx` = you asked wrong, `5xx` = the server broke.
3. Four verbs: **GET** read, **POST** create, **PUT** update, **DELETE** remove - all the same call/`.status_code`/`.json()` shape.

**You also met the OpenAI API** and saw it is just an HTTP POST with an `Authorization: Bearer` header - the same skills, plus a key. Streaming there (`data: ...` pieces) is a sneak peek at the async/streaming days.

**Next - Day 10 (Async):** we saw five slow calls take ~2.5 seconds, one after another. Tomorrow we make many slow calls run *together*, so they finish in the time of one - using these same real endpoints, no pretend waiting.
